<a href="https://colab.research.google.com/github/AllyApitchaya/msc-adr-prediction/blob/main/notebooks/05_biobert_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/MSc_ADR_Project'
print(f"Project path: {PROJECT_PATH}\n")

# BioBERT is a large model; a GPU is required for reasonable speed.
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime "
          "type > T4 GPU, then re-run.")

Mounted at /content/drive
Project path: /content/drive/MyDrive/MSc_ADR_Project

GPU available: Tesla T4


In [2]:
# Install the Hugging Face transformers library.
# Colab has torch already; we just need transformers + tokenizers.
!pip install -q transformers==4.44.2

import transformers
from transformers import AutoTokenizer, AutoModel

print(f"transformers version: {transformers.__version__}")
print("Imports OK.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.0 MB/s eta 0:00:00
transformers version: 4.44.2
Imports OK.


In [3]:
# Load BioBERT v1.1 — a BERT model pre-trained on biomedical text
# (PubMed abstracts + PMC articles). We use the version hosted by DMIS Lab,
# the original authors (Lee et al., 2020).
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"

print(f"Loading: {MODEL_NAME}")
print("(first run downloads ~420 MB — this may take a minute)\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

# Move the model to the GPU and switch to evaluation mode.
# eval() disables dropout — we are extracting features, not training.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

# Count parameters to confirm we loaded the full model.
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded on: {device}")
print(f"Parameters: {n_params:,} (~110M expected for BERT-base)")

Loading: dmis-lab/biobert-base-cased-v1.1
(first run downloads ~420 MB — this may take a minute)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Model loaded on: cuda
Parameters: 108,310,272 (~110M expected for BERT-base)


In [4]:
# Before embedding, let's SEE how BioBERT breaks an ADR term into tokens.
# We pick a real ADR term from our dataset as the example.
example_adr = "Lactic acidosis"

# Tokenize: convert text -> token IDs the model understands.
encoded = tokenizer(example_adr, return_tensors="pt")

# Show what happened, step by step.
token_ids = encoded["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(token_ids)

print(f"Original ADR term : '{example_adr}'")
print(f"Number of tokens  : {len(tokens)}")
print()
print(f"{'token':<16} {'id':>8}")
print("-" * 26)
for tok, tid in zip(tokens, token_ids.tolist()):
    print(f"{tok:<16} {tid:>8}")

Original ADR term : 'Lactic acidosis'
Number of tokens  : 6

token                  id
--------------------------
[CLS]                 101
la                   2495
##ctic              11143
acid                 5190
##osis              11776
[SEP]                 102


In [5]:
# Now the real test: turn an ADR term into a single fixed-length vector.
# This is the function we will reuse in Cell 6 for all ADR terms.

def embed_adr(text):
    """Convert one ADR term into a 768-dim vector using BioBERT's [CLS] token."""
    # Tokenize and move inputs to the same device as the model.
    encoded = tokenizer(text, return_tensors="pt").to(device)

    # no_grad: we are not training, so skip gradient tracking (saves memory).
    with torch.no_grad():
        output = model(**encoded)

    # last_hidden_state shape: (batch=1, num_tokens, hidden=768)
    # We take the [CLS] token (position 0) as the whole-term summary.
    cls_vector = output.last_hidden_state[0, 0, :]
    return cls_vector.cpu()

# Test on the same ADR term.
example_adr = "Lactic acidosis"
vec = embed_adr(example_adr)

print(f"ADR term       : '{example_adr}'")
print(f"Embedding shape: {tuple(vec.shape)}")
print(f"Data type      : {vec.dtype}")
print()
print(f"First 8 values : {vec[:8].tolist()}")

ADR term       : 'Lactic acidosis'
Embedding shape: (768,)
Data type      : torch.float32

First 8 values : [0.22696322202682495, 0.01837378740310669, -0.2509250044822693, -0.13474220037460327, -0.3618948757648468, -0.17141243815422058, 0.03491605445742607, 0.09490121155977249]


In [6]:
import pandas as pd
import os

# Load the preprocessed splits and collect EVERY unique ADR term across them.
# We embed across all splits because notebook 06 needs vectors for any ADR
# it might see in train, val, or test.
processed_dir = os.path.join(PROJECT_PATH, "data", "processed")

splits = {}
for name in ["train", "val", "test"]:
    splits[name] = pd.read_csv(os.path.join(processed_dir, f"{name}.csv"))
    print(f"{name:6s}: {len(splits[name]):,} rows")

all_adrs = pd.concat([s["adr"] for s in splits.values()])
unique_adrs = sorted(all_adrs.dropna().unique())
print(f"\nUnique ADR terms to embed: {len(unique_adrs):,}")

train : 8,240 rows
val   : 4,912 rows
test  : 9,016 rows

Unique ADR terms to embed: 3,576


In [7]:
import numpy as np
from tqdm.auto import tqdm

# Embed all unique ADR terms in BATCHES (much faster than one-by-one).
# Batching lets the GPU process many terms in parallel.
BATCH_SIZE = 32

def embed_batch(texts):
    """Embed a list of ADR terms -> array of shape (len(texts), 768)."""
    # padding=True makes every term the same length within the batch;
    # truncation caps very long terms (ADR names are short, so rarely triggered).
    encoded = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=64
    ).to(device)

    with torch.no_grad():
        output = model(**encoded)

    # Take the [CLS] token (position 0) for every term in the batch.
    cls_vectors = output.last_hidden_state[:, 0, :]
    return cls_vectors.cpu().numpy()

# Process all unique ADR terms.
all_vectors = []
for i in tqdm(range(0, len(unique_adrs), BATCH_SIZE), desc="Embedding ADRs"):
    batch = unique_adrs[i : i + BATCH_SIZE]
    all_vectors.append(embed_batch(batch))

# Stack into one big array: (3576, 768)
adr_embeddings = np.vstack(all_vectors)

print(f"\nDone.")
print(f"Embedding matrix shape: {adr_embeddings.shape}")
print(f"Expected             : ({len(unique_adrs)}, 768)")

Embedding ADRs:   0%|          | 0/112 [00:00<?, ?it/s]


Done.
Embedding matrix shape: (3576, 768)
Expected             : (3576, 768)


In [8]:
# Save the embeddings so notebook 06 can load them WITHOUT re-running BioBERT.
# We save two aligned files:
#   - the ADR term list (the "row labels")
#   - the embedding matrix itself
# Row i of the matrix corresponds to unique_adrs[i].

embeddings_dir = os.path.join(PROJECT_PATH, "data", "embeddings")
os.makedirs(embeddings_dir, exist_ok=True)

# Save the matrix as a compressed .npz, keyed so it is self-documenting.
matrix_path = os.path.join(embeddings_dir, "adr_biobert_embeddings.npz")
np.savez_compressed(
    matrix_path,
    adr_terms=np.array(unique_adrs, dtype=object),
    embeddings=adr_embeddings,
)

# Quick reload check — confirm the file is readable and aligned.
loaded = np.load(matrix_path, allow_pickle=True)
print(f"Saved to: {matrix_path}\n")
print(f"  adr_terms  : {loaded['adr_terms'].shape}")
print(f"  embeddings : {loaded['embeddings'].shape}")
print()
print(f"Sanity check — row 0:")
print(f"  ADR term : '{loaded['adr_terms'][0]}'")
print(f"  vector   : first 5 = {loaded['embeddings'][0, :5].round(4).tolist()}")
print("\nNotebook 05 complete. BioBERT embeddings ready for notebook 06.")

Saved to: /content/drive/MyDrive/MSc_ADR_Project/data/embeddings/adr_biobert_embeddings.npz

  adr_terms  : (3576,)
  embeddings : (3576, 768)

Sanity check — row 0:
  ADR term : '5-hydroxyindolacetic acid in urine increased'
  vector   : first 5 = [0.01940000057220459, -0.1679999977350235, -0.2782000005245209, -0.0869000032544136, -0.23849999904632568]

Notebook 05 complete. BioBERT embeddings ready for notebook 06.
